# Significance Testing and Multicollinearity in Multiple Regression

## 1. Introduction and Overview
In simple linear regression, inference is straightforward because there is only one predictor. In multiple regression, we face two distinct inferential questions:

1. **Is the model useful overall?** (Answered by the F-Test)
2. **Which specific predictors matter?** (Answered by Individual t-Tests)

However, when predictors are highly correlated with each other, we run into one of the most critical challenges in statistical modeling: **Multicollinearity**. This notebook explores how to test for significance, how multicollinearity breaks these tests, and how to diagnose and fix it.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries successfully imported! Environment is ready.")

## 3. Data Creation: Synthetic Employee Salary Dataset
To explore these concepts, we will generate a synthetic dataset of employee salaries. 
Initially, we will use two independent predictors: `Years_Experience` and `Performance_Score`.

Later, we will introduce a third variable that is highly correlated with `Years_Experience` to demonstrate Multicollinearity.

In [ ]:
# Generate independent predictors
n_employees = 300

years_experience = np.random.uniform(1, 20, size=n_employees)
performance_score = np.random.normal(70, 10, size=n_employees)

# True Salary Formula (Base: 40k + 3k per year experience + 0.5k per performance point)
base_salary = 40
noise = np.random.normal(0, 5, size=n_employees)

salary_k = base_salary + (3.0 * years_experience) + (0.5 * performance_score) + noise

# Combine into a Pandas DataFrame
df_clean = pd.DataFrame({
    'Experience': years_experience,
    'Performance': performance_score,
    'Salary': salary_k
})

print(f"Synthetic dataset created with {n_employees} employees.")
print("\n--- First 5 rows of Clean Dataset ---")
print(df_clean.head().round(2))

## 4. Core Concept 1: Overall Model Significance (The F-Test)
The F-test evaluates whether the regression model explains a statistically meaningful amount of variation in Y. It tests the predictive power of the entire model collectively.

**Hypotheses of the F-Test:**
- **Null Hypothesis (H0):** beta_1 = beta_2 = ... = beta_k = 0 (The Useless Model Hypothesis)
- **Alternative Hypothesis (Ha):** At least one beta_j != 0 (The model explains some variation)

The F-statistic is a ratio of Mean Square Regression (MSR) to Mean Square Error (MSE). 
F = MSR / MSE.

In [ ]:
# Fit the clean multiple regression model
model_clean = smf.ols('Salary ~ Experience + Performance', data=df_clean).fit()

# Extract F-statistic and its p-value
f_stat = model_clean.fvalue
f_pvalue = model_clean.f_pvalue

print("--- Overall Model Significance (F-Test) ---")
print(f"F-Statistic: {f_stat:.2f}")
print(f"F-Test P-value: {f_pvalue:.4e}")

if f_pvalue < 0.05:
    print("\nConclusion: Reject the Null Hypothesis.")
    print("The model is highly significant overall. At least one predictor contributes to Salary.")
else:
    print("\nConclusion: Fail to reject the Null Hypothesis.")
    print("The model does not explain a significant amount of variation.")

## 5. Core Concept 2: Individual Predictor Significance (t-Tests)
Once we know the overall model is significant, we use individual t-tests to answer: *Which specific predictors matter?*

**Hypotheses for Individual t-Tests:**
- **H0:** beta_j = 0 (Predictor j has no unique linear effect after controlling for others)
- **Ha:** beta_j != 0 (Predictor j contributes uniquely)

The t-statistic is a signal-to-noise ratio: t = Estimated Coefficient / Standard Error.

In [ ]:
print("--- Individual Predictor Significance (t-Tests) ---")
# Extracting the coefficients summary table
summary_table = model_clean.summary().tables[1]
print(summary_table)

print("\nInterpretation:")
print("- The p-values (P>|t|) for both Experience and Performance are near 0.000.")
print("- This means both predictors contribute unique, statistically significant explanatory power.")

## 6. The Threat: Introducing Multicollinearity
What happens if we introduce a variable that measures almost the exact same thing as an existing variable? 

We will add a new column called `Seniority_Level`. Suppose Seniority is directly calculated from Years of Experience (e.g., Seniority = Experience * 1.5 + some small random noise). 

This creates the **Twin Predictors Problem**.

In [ ]:
# Create the correlated variable
df_multi = df_clean.copy()
df_multi['Seniority_Level'] = (df_multi['Experience'] * 1.5) + np.random.normal(0, 1.5, size=n_employees)

print("--- New Dataset with Multicollinearity ---")
print(df_multi[['Experience', 'Seniority_Level']].head())

# Check the pairwise correlation
corr_val = df_multi['Experience'].corr(df_multi['Seniority_Level'])
print(f"\nCorrelation between Experience and Seniority: {corr_val:.4f}")
print("This extremely high correlation (> 0.95) guarantees severe multicollinearity.")

## 7. The Classic Symptom: Significant F-Test, Insignificant t-Tests
Let's fit a new model using all three predictors. 
Watch what happens to the standard errors and p-values of the correlated variables. The model knows they collectively matter (F-test is significant), but it cannot isolate which one deserves the credit (t-tests become insignificant).

In [ ]:
# Fit the model with multicollinearity
model_multi = smf.ols('Salary ~ Experience + Performance + Seniority_Level', data=df_multi).fit()

print("--- Model WITH Multicollinearity ---")
print(f"Overall F-Test P-value: {model_multi.f_pvalue:.4e} (Still highly significant!)")
print("\n--- Individual t-Tests ---")
print(model_multi.summary().tables[1])

print("\n--- Diagnostic Observation ---")
print("Notice that the p-values for Experience and Seniority_Level are suddenly much higher (possibly > 0.05)!")
print("The Standard Errors (std err) have inflated massively compared to the clean model.")
print("The coefficients themselves are unstable and drastically altered.")

## 8. Diagnosing Multicollinearity: Correlation Matrix
The first step in diagnosing multicollinearity is checking pairwise correlations using a heatmap. 
Values where |r| > 0.8 are warning signs.

In [ ]:
plt.figure(figsize=(8, 6))
corr_matrix = df_multi[['Salary', 'Experience', 'Performance', 'Seniority_Level']].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={"size": 12})
plt.title('Correlation Matrix (Detecting Multicollinearity)', fontsize=14)
plt.tight_layout()
plt.show()

print("The deep red square between Experience and Seniority_Level immediately highlights the problem.")

## 9. Diagnosing Multicollinearity: Variance Inflation Factor (VIF)
Correlation matrices only show pairwise relationships. They can miss multivariable collinearity. 
The gold standard diagnostic is the **Variance Inflation Factor (VIF)**.

VIF = 1 / (1 - Rj_squared)

**Interpretation Rules:**
- VIF = 1: No multicollinearity
- VIF 1-5: Mild/Moderate
- VIF > 5: Potentially problematic
- VIF > 10: Severe multicollinearity

In [ ]:
# Calculate VIF for each predictor
# VIF requires a constant column to compute properly in statsmodels
X_multi = df_multi[['Experience', 'Performance', 'Seniority_Level']]
X_multi_with_const = sm.add_constant(X_multi)

vif_data = pd.DataFrame()
vif_data['Predictor'] = X_multi_with_const.columns

# Compute VIF using list comprehension
vif_data['VIF_Score'] = [variance_inflation_factor(X_multi_with_const.values, i) 
                         for i in range(X_multi_with_const.shape[1])]

# Remove the constant from the display as it's not a predictor
vif_data = vif_data[vif_data['Predictor'] != 'const'].reset_index(drop=True)

print("--- Variance Inflation Factor (VIF) Results ---")
print(vif_data)
print("\nConclusion: Experience and Seniority_Level have VIF scores far above 10.")
print("This confirms severe multicollinearity inflating their coefficient variances.")

## 10. Remedy 1: Dropping Redundant Variables
The simplest and most common practical fix is to drop one of the highly correlated predictors. 
This forces the model to assign all the shared explanatory credit to the remaining variable, stabilizing the geometry and standard errors.

In [ ]:
# We drop the redundant 'Seniority_Level' variable
model_fixed = smf.ols('Salary ~ Experience + Performance', data=df_multi).fit()

print("--- Model AFTER Dropping Seniority_Level ---")
print(model_fixed.summary().tables[1])

print("\nNotice:")
print("- The Standard Error for Experience has dropped drastically.")
print("- The p-value is significant again.")
print("- The coefficient returned to its true value (~3.0).")

## 11. Visualizing Coefficient Instability (Standard Error Inflation)
To truly understand the damage multicollinearity does, let's visualize the Standard Errors of the 'Experience' coefficient before and after fixing the model.

In [ ]:
# Extract Standard Errors
se_collinear = model_multi.bse['Experience']
se_fixed = model_fixed.bse['Experience']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['With Multicollinearity', 'After Dropping Variable'], [se_collinear, se_fixed], color=['red', 'green'])

ax.set_ylabel('Standard Error of Experience Coefficient', fontsize=12)
ax.set_title('Standard Error Inflation due to Multicollinearity', fontsize=14)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=12)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Remedy 2: Ridge Regression (Advanced)
Sometimes we cannot drop variables (e.g., if domain knowledge requires all features). 
Ridge Regression adds a penalty term (L2 regularization) that shrinks coefficients, preventing the unstable inversions caused by collinearity.

Note: Ridge requires features to be scaled because the penalty affects coefficients based on their magnitude.

In [ ]:
# Prepare features and target
X_features = df_multi[['Experience', 'Performance', 'Seniority_Level']]
y_target = df_multi['Salary']

# Standardize the features (Crucial for Ridge)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Fit Ridge Regression
ridge_model = Ridge(alpha=10.0) # alpha is the penalty term lambda
ridge_model.fit(X_scaled, y_target)

print("--- Ridge Regression Coefficients ---")
for col, coef in zip(X_features.columns, ridge_model.coef_):
    print(f"{col}: {coef:.3f}")

print("\nUnlike OLS, Ridge shrinks the overlapping coefficients (Experience and Seniority) ")
print("to stabilize them, preventing them from swinging wildly in opposite directions.")

## 13. Practice Exercises: House Price Modeling
**Scenario:** You are modeling House Prices. You have variables: 
`Bedrooms`, `Square_Feet_Living`, and `Square_Feet_Total` (which includes garage and patio).

Naturally, `Square_Feet_Living` and `Square_Feet_Total` are highly correlated. 
Let's generate this dataset and test your skills.

In [ ]:
# Generate Practice Data
n_houses = 250
bedrooms = np.random.randint(1, 6, size=n_houses)
sqft_living = bedrooms * 500 + np.random.normal(0, 200, size=n_houses)
sqft_total = sqft_living * 1.3 + np.random.normal(0, 100, size=n_houses) # Highly correlated

# True Price: Base 50k + 20k/bedroom + 150/sqft_living
price_k = 50 + (20 * bedrooms) + (0.15 * sqft_living) + np.random.normal(0, 30, size=n_houses)

df_house = pd.DataFrame({
    'Bedrooms': bedrooms,
    'Sqft_Living': sqft_living,
    'Sqft_Total': sqft_total,
    'Price': price_k
})

print("Practice House Data Generated.")
print(df_house.head())

### Exercise 1: Fit Model and Check VIF
1. Fit an OLS model predicting `Price` using all three predictors.
2. Print the Summary Table to check the t-tests.
3. Calculate and print the VIF for all predictors.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
# 1. Fit Model
house_model_bad = smf.ols('Price ~ Bedrooms + Sqft_Living + Sqft_Total', data=df_house).fit()
print("--- House Model with all Predictors ---")
print(house_model_bad.summary().tables[1])

# 2. Calculate VIF
X_house = df_house[['Bedrooms', 'Sqft_Living', 'Sqft_Total']]
X_house_const = sm.add_constant(X_house)

vif_house = pd.DataFrame()
vif_house['Feature'] = X_house_const.columns
vif_house['VIF'] = [variance_inflation_factor(X_house_const.values, i) for i in range(X_house_const.shape[1])]
vif_house = vif_house[vif_house['Feature'] != 'const'].reset_index(drop=True)

print("\n--- VIF Results ---")
print(vif_house)

### Exercise 2: Resolve Multicollinearity
1. Drop the `Sqft_Total` variable.
2. Refit the OLS model.
3. Print the new summary and note the change in standard errors.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
# 1 & 2. Refit dropping Sqft_Total
house_model_good = smf.ols('Price ~ Bedrooms + Sqft_Living', data=df_house).fit()

print("--- Fixed House Model ---")
print(house_model_good.summary().tables[1])

print("\nExplanation:")
print("By removing the redundant variable, the standard error for Sqft_Living stabilized,")
print("and the model correctly attributes the price effect to the true underlying factors.")

## 14. Visualization Gallery: The Geometry of Collinearity
When predictors are highly correlated, they form a tight line rather than spreading out into a plane. This makes the regression plane wobbly (like a table with legs placed too close together in a straight line).
Let's visualize the predictor space of the correlated variables.

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

# Plot the highly correlated predictors from the House dataset
sc = ax.scatter(df_house['Sqft_Living'], df_house['Sqft_Total'], df_house['Price'], 
                c=df_house['Price'], cmap='viridis', alpha=0.8)

ax.set_xlabel('Square Feet Living')
ax.set_ylabel('Square Feet Total')
ax.set_zlabel('Price')
ax.set_title('Collinear Predictor Space: Notice the "Line-like" Data Spread', fontsize=12)

plt.colorbar(sc, label='Price', pad=0.1)
plt.show()

print("Because Sqft_Living and Sqft_Total form a narrow diagonal band on the XY plane,")
print("the model struggles to tilt the regression plane reliably. This is geometric instability.")

## 15. Summary and Key Takeaways

- **Overall Significance (F-Test)**: Tests if the model as a whole is useful (H0: all slopes = 0). A low p-value means the model predicts better than random noise.
- **Individual Significance (t-Test)**: Tests if a specific predictor adds unique value after controlling for everything else in the model.
- **Multicollinearity**: Occurs when predictors are highly correlated, sharing overlapping information.
- **The Classic Symptom**: A highly significant overall F-test, but insignificant individual t-tests.
- **The Damage**: Multicollinearity violently inflates Standard Errors and makes coefficients mathematically unstable.
- **Diagnosis**: Use Correlation Matrices for pairs, and the Variance Inflation Factor (VIF) for multi-variable collinearity (VIF > 5 is a warning).
- **The Fix**: The simplest and most robust fix is to drop redundant variables. Alternatively, use Ridge Regression to regularize and shrink coefficients.